# AgentCore Gateway Demo — Centralized MCP with Auth Propagation

This notebook demonstrates **centralized MCP server management** with **Okta OIDC auth propagation** using AWS AgentCore Gateway.

## The "whoa" moments

1. **One Gateway endpoint** manages both Lambda functions AND MCP servers
2. **User identity flows through** — each user's JWT is validated by the Gateway
3. **Different users, different tools** — Alice (analyst) sees only weather tools; Bob (finance-admin) sees both
4. **Gateway-level ENFORCE** — Cedar policies block unauthorized tool calls at the Gateway, not the agent

## Prerequisites

Run `01_setup.ipynb` first to create the infrastructure.

## Cell 1: Load Config + Pre-Authenticate Both Users

We authenticate both Alice and Bob upfront via Okta Resource Owner Password flow, then create separate Strands agents for each — one connected to the Gateway with Alice's token, one with Bob's.

The Gateway validates each JWT against Okta's JWKS endpoint and checks:
- **Audience** matches `api://default`
- **Client ID** matches the configured `allowedClients`
- **Signature** is valid per Okta's signing keys

Cedar policies in **ENFORCE mode** then control which tools each user can discover and invoke:
- `AgentCore::OAuthUser` (all) → WeatherTools: **ALLOW**
- `AgentCore::OAuthUser::"bob@..."` → FinanceTools: **ALLOW**
- Everyone else → FinanceTools: **DENY** (no matching permit)

In [ ]:
import os
import json
import jwt
import httpx
import requests
from dotenv import load_dotenv
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

# --- Load Gateway config from setup notebook ---
with open("gateway_config.json") as f:
    config = json.load(f)

GATEWAY_URL = config["gateway_url"]
OKTA_DOMAIN = config["okta_domain"]
OKTA_CLIENT_ID = config["okta_client_id"]
OKTA_ISSUER = config["okta_issuer"]
OKTA_CLIENT_SECRET = os.environ["OKTA_CLIENT_SECRET"]

# --- Okta credentials for test users ---
ALICE_USERNAME = os.environ["ALICE_USERNAME"]
ALICE_PASSWORD = os.environ["ALICE_PASSWORD"]
BOB_USERNAME = os.environ["BOB_USERNAME"]
BOB_PASSWORD = os.environ["BOB_PASSWORD"]

# --- Model to use for Strands agents ---
MODEL_ID = "global.anthropic.claude-sonnet-4-20250514-v1:0"


def get_okta_token(username: str, password: str) -> dict:
    """Authenticate a user via Okta Resource Owner Password flow and return the JWT."""
    token_url = f"{OKTA_ISSUER}/v1/token"
    response = requests.post(
        token_url,
        data={
            "grant_type": "password",
            "username": username,
            "password": password,
            "scope": "openid profile groups",
            "client_id": OKTA_CLIENT_ID,
            "client_secret": OKTA_CLIENT_SECRET,
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
    )
    response.raise_for_status()
    token_data = response.json()
    access_token = token_data["access_token"]

    # Decode JWT (without verification, just to display claims)
    claims = jwt.decode(access_token, options={"verify_signature": False})
    return {"access_token": access_token, "claims": claims}


# --- Pre-authenticate both users ---
print("Authenticating Alice (analyst group)...")
alice_auth = get_okta_token(ALICE_USERNAME, ALICE_PASSWORD)
print(f"  User: {alice_auth['claims'].get('sub', 'unknown')}")
print(f"  Groups: {alice_auth['claims'].get('groups', [])}")

print("\nAuthenticating Bob (finance-admins group)...")
bob_auth = get_okta_token(BOB_USERNAME, BOB_PASSWORD)
print(f"  User: {bob_auth['claims'].get('sub', 'unknown')}")
print(f"  Groups: {bob_auth['claims'].get('groups', [])}")

# --- Create MCPClients connected to Gateway with each user's token ---
alice_http = httpx.AsyncClient(headers={"Authorization": f"Bearer {alice_auth['access_token']}"})
bob_http = httpx.AsyncClient(headers={"Authorization": f"Bearer {bob_auth['access_token']}"})

alice_client = MCPClient(
    lambda: streamable_http_client(GATEWAY_URL, http_client=alice_http),
    startup_timeout=60,
)

bob_client = MCPClient(
    lambda: streamable_http_client(GATEWAY_URL, http_client=bob_http),
    startup_timeout=60,
)

# --- Create Strands Agents ---
# No agent-level RBAC needed — Cedar policies in ENFORCE mode handle access control
# at the Gateway level. The agent simply calls tools; the Gateway decides what's allowed.
alice_client.__enter__()
bob_client.__enter__()

alice_tools = alice_client.list_tools_sync()
bob_tools = bob_client.list_tools_sync()

SYSTEM_PROMPT = """You are a helpful assistant.
If a tool call fails with an access denied or policy enforcement error, explain clearly
that the user does not have permission to use that tool."""

alice_agent = Agent(
    model=MODEL_ID,
    tools=alice_tools,
    system_prompt=SYSTEM_PROMPT,
)

bob_agent = Agent(
    model=MODEL_ID,
    tools=bob_tools,
    system_prompt=SYSTEM_PROMPT,
)

alice_tool_names = [t.tool_name if hasattr(t, 'tool_name') else str(t) for t in alice_tools]
bob_tool_names = [t.tool_name if hasattr(t, 'tool_name') else str(t) for t in bob_tools]

print(f"\n--- Gateway Authentication (ENFORCE mode) ---")
print(f"  Gateway URL: {GATEWAY_URL}")
print(f"  Alice tools: {alice_tool_names}")
print(f"  Bob tools:   {bob_tool_names}")
print(f"\n  Alice sees {len(alice_tools)} tool(s), Bob sees {len(bob_tools)} tool(s)")
print(f"  Access control enforced by Cedar policies at the Gateway")

## Cell 2: Show Available Tools Per User

With ENFORCE mode, the Gateway filters tool discovery based on Cedar policies.
Alice only sees tools she's authorized to use; Bob sees a broader set.

In [ ]:
print("Tools visible through AgentCore Gateway:\n")

print("ALICE (analyst):")
for tool in alice_tools:
    name = tool.tool_name if hasattr(tool, 'tool_name') else str(tool)
    print(f"  - {name}")
print(f"  Total: {len(alice_tools)} tool(s)\n")

print("BOB (finance-admin):")
for tool in bob_tools:
    name = tool.tool_name if hasattr(tool, 'tool_name') else str(tool)
    print(f"  - {name}")
print(f"  Total: {len(bob_tools)} tool(s)\n")

print("Cedar ENFORCE mode filters tool discovery per user.")
print("Alice cannot even see FinanceTools — the Gateway hides them.")

## Cell 3: Alice Asks for Weather (ALLOWED)

Alice is in the `analysts` group. The Cedar policy allows **all authenticated users** to access `WeatherTools`. This should succeed.

In [8]:
print("=" * 60)
print("ALICE (analyst) asks: 'What's the weather in Sydney?'")
print("=" * 60)

response = alice_agent("What's the weather in Sydney?")

print("\n" + "=" * 60)
print("RESULT: Alice (analyst) CAN access weather data")
print("Cedar policy: WeatherTools → ALLOW all authenticated users")
print("=" * 60)

ALICE (analyst) asks: 'What's the weather in Sydney?'
I'll get the current weather information for Sydney.
Tool #1: WeatherTools___get_weather
The current weather in Sydney is:
- **Temperature:** 22°C
- **Condition:** Partly cloudy
- **Humidity:** 65%
- **Wind:** 18 km/h

It's a pleasant day in Sydney with mild temperatures and partly cloudy skies!
RESULT: Alice (analyst) CAN access weather data
Cedar policy: WeatherTools → ALLOW all authenticated users


## Cell 4: Alice Asks for Finance Data (BLOCKED by Gateway)

Alice asks for confidential revenue data. The Cedar policy only permits Bob's `sub` to access FinanceTools.
The Gateway **enforces** this — Alice's agent doesn't even have `FinanceTools___get_revenue_data` in its tool list.

**This is the key "whoa" moment — the Gateway enforces access control, not the LLM.**

In [ ]:
print("=" * 60)
print("ALICE (analyst) asks: 'Show me the quarterly revenue data'")
print("=" * 60)

response = alice_agent("Show me the quarterly revenue data")

print("\n" + "=" * 60)
print("RESULT: Alice CANNOT access finance data!")
print(f"Alice's available tools: {alice_tool_names}")
print("Gateway Cedar policy: FinanceTools not in Alice's permitted actions")
print("The Gateway doesn't even expose FinanceTools to Alice's agent")
print("=" * 60)

## Cell 5: Bob Asks for Finance Data (ALLOWED by Gateway)

Now Bob asks the **exact same question**. His JWT `sub` matches the Cedar policy for FinanceTools.
The Gateway permits the call and returns the data.

**Same question, different user, different result — enforced by Gateway Cedar policies, not the LLM.**

In [ ]:
print("=" * 60)
print("BOB (finance-admin) asks: 'Show me the quarterly revenue data'")
print("=" * 60)

response = bob_agent("Show me the quarterly revenue data")

print("\n" + "=" * 60)
print("RESULT: Bob (finance-admin) CAN access finance data!")
print(f"Bob's available tools: {bob_tool_names}")
print("Same question as Alice. Same Gateway. Same endpoint.")
print("The ONLY difference: Bob's JWT sub matches the Cedar policy")
print("=" * 60)

## Cell 6: Architecture Summary

What we just demonstrated:

In [ ]:
print("""
==================================================================
          AgentCore Gateway Demo - Summary
==================================================================

  ARCHITECTURE:

    User (CLI)
      |
      | username + password (Resource Owner Password grant)
      v
    Okta Token Endpoint --> JWT (with groups, client_id claims)
      |
      v
    Strands Agent (JWT as Bearer token)
      |
      | MCP (StreamableHTTP + Bearer JWT)
      v
    AgentCore Gateway
      |-- Ingress Auth: JWT validation (signature, audience, client_id)
      |-- Cedar Policy Engine (ENFORCE mode)
      |     principal: AgentCore::OAuthUser::"<JWT sub>"
      |     action:    AgentCore::Action::"<TargetName>"
      |     resource:  AgentCore::Gateway::"<gateway-arn>"
      |
      +------------------+
      |                  |
      v                  v
    Lambda            Lambda
   (Weather)         (Finance)

  AUTH FLOW:
    1. User authenticates via Okta ROPC grant -> JWT with sub claim
    2. Strands Agent attaches JWT as Bearer token on MCP connection
    3. Gateway validates JWT via Okta JWKS endpoint
    4. Cedar policies evaluated in ENFORCE mode
    5. Gateway permits or denies tool discovery and invocation
    6. No access control logic needed in the agent or MCP servers

  CEDAR POLICIES:
    WeatherTools -> ALL AgentCore::OAuthUser -> ALLOW
    FinanceTools -> AgentCore::OAuthUser::"bob@..." -> ALLOW
    FinanceTools -> everyone else -> DENY (no matching permit)

  WHAT WE DEMONSTRATED:
    Alice (analyst)      -> Weather data  -> ALLOWED (Gateway permit)
    Alice (analyst)      -> Finance data  -> BLOCKED (Gateway deny)
    Bob (finance-admin)  -> Finance data  -> ALLOWED (Gateway permit)

  KEY VALUE:
    - ONE Gateway manages multiple Lambda-backed MCP tools
    - Direct Okta OIDC -- no Cognito intermediary
    - Cedar policies ENFORCE access at the Gateway level
    - User identity (JWT sub) mapped to Cedar principals
    - Agent needs zero access control logic
    - Zero auth code in MCP servers

==================================================================
""")

# Cleanup MCPClients and httpx clients
alice_client.__exit__(None, None, None)
bob_client.__exit__(None, None, None)
print("Agents disconnected. Run cleanup in 01_setup.ipynb when done.")